Instalar Librerias 

In [1]:
pip install -r requirements.txt

   ---------------------------------------- 0.0/1.2 MB ? eta -:--:--
   --------- ------------------------------ 0.3/1.2 MB ? eta -:--:--
   ------------------------------------ --- 1.0/1.2 MB 3.6 MB/s eta 0:00:01
   ---------------------------------------- 1.2/1.2 MB 3.4 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Librerias

In [1]:
import math
import pandas as pd
import osmnx as ox
import networkx as nx
import re
import psycopg2
from geopy.distance import geodesic
import tracemalloc
import folium
import webbrowser
import os
from collections import defaultdict
from shapely.geometry import Point, LineString
import geopandas as gpd
import numpy as np
# Al inicio de tu archivo, junto con los otros imports
from http.server import HTTPServer, SimpleHTTPRequestHandler
import threading

Exportar los datos del CSV y definir la Latitud y Longitud

In [45]:
# --- 1️⃣ CONFIGURACIÓN DE CONEXIÓN REMOTA (RAILWAY) ---
DB_CONFIG = {
    'host': 'tramway.proxy.rlwy.net',
    'port': 31631,
    'dbname': 'railway',
    'user': 'postgres',
    'password': 'KAGJhRklTcsevGqKEgCNPfmdDiGzsLyQ'
}

# --- 2️⃣ CONSULTA SQL ---
# Cambia el esquema si no es 'public'
QUERY = "SELECT * FROM public.ongs;"

# --- 3️⃣ CONECTAR Y LEER DATOS ---
try:
    conn = psycopg2.connect(**DB_CONFIG)
    df_raw = pd.read_sql(QUERY, conn)
    conn.close()
    print("✅ Datos cargados exitosamente desde PostgreSQL")
except Exception as e:
    raise RuntimeError(f"❌ Error al conectar o leer la base de datos: {e}")

print("Encabezados detectados:", list(df_raw.columns))

# --- 4️⃣ Normalización de nombres de columnas ---
def norm_col(c):
    return re.sub(r'[^a-z0-9]', '', c.lower(), flags=re.IGNORECASE)

col_map = {norm_col(c): c for c in df_raw.columns}

# Alias posibles
name_aliases = ['nombre','name','org','organization','ong','institucion','institución','nomong','nom_ong']
type_aliases = ['tipo','type','categoria','category']
lat_aliases = ['latitud','latitude','lat','y']
lon_aliases = ['longitud','longitude','lon','lng','x']

def find_column(alias_list):
    for a in alias_list:
        a_norm = re.sub(r'[^a-z0-9]', '', a.lower())
        if a_norm in col_map:
            return col_map[a_norm]
    return None

col_name = find_column(name_aliases)
col_type = find_column(type_aliases)
col_lat  = find_column(lat_aliases)
col_lon  = find_column(lon_aliases)

if col_lat is None or col_lon is None:
    raise ValueError("No se encontraron columnas de latitud/longitud en la tabla 'ongs'.")

# --- 5️⃣ Construir DataFrame normalizado ---
df = df_raw.copy()
df_norm = pd.DataFrame()
df_norm['name'] = df[col_name] if col_name is not None else ''
df_norm['type'] = df[col_type] if col_type is not None else ''

def to_float_series(s):
    s2 = s.astype(str).str.replace(',', '.').str.strip()
    return pd.to_numeric(s2, errors='coerce')

df_norm['lat'] = to_float_series(df[col_lat])
df_norm['lon'] = to_float_series(df[col_lon])

# --- 6️⃣ Eliminar filas sin coordenadas válidas ---
before = len(df_norm)
df_norm = df_norm.dropna(subset=['lat','lon']).reset_index(drop=True)
after = len(df_norm)

print(f"Filas leídas: {before}, filas con coordenadas válidas: {after}")

# --- 7️⃣ Inspección rápida ---
display(df_norm.head(10))
print(df_norm.dtypes)

# --- 8️⃣ Construir lista de waypoints ---
waypoints = [
    {'name': row['name'], 'type': row['type'], 'lat': float(row['lat']), 'lon': float(row['lon'])}
    for _, row in df_norm.iterrows()
]

print(f"Waypoints construidos: {len(waypoints)}. Primeros 10 (si existen):")
for i, w in enumerate(waypoints[:10]):
    print(i+1, w)

C:\Users\samic\AppData\Local\Temp\ipykernel_19980\1223187190.py:17: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_raw = pd.read_sql(QUERY, conn)


✅ Datos cargados exitosamente desde PostgreSQL
Encabezados detectados: ['id_ong', 'id_municipio', 'nom_ong', 'tipo', 'latitud', 'longitud']
Filas leídas: 147, filas con coordenadas válidas: 147


,name,type,lat,lon
0,San Juan Diego,Albergue,19.843352,-99.190905
1,Casa del Migrante en Tijuana,Albergue,32.522948,-117.001793
2,Casa del Peregrino Migrante,Albergue,20.368514,-99.692361
3,San Juan Diego y San Francisco de Asís,Albergue,25.837029,-97.486004
4,Hidalgo (Tamaulipas),Frontera,26.083200,-98.277300
5,Templo Embajadores de Jesús,Albergue,32.507857,-117.079708
6,Desayunador Padre Chava,Comedor,32.536756,-117.032126
7,Sembrando Esperanza,Albergue,22.764085,-102.658088
8,Hotel de los Migrantes Ángeles sin Fronteras,Albergue,32.662699,-115.495064
9,Cáritas Mazatlán,Albergue,23.254298,-106.426815


name     object
type     object
lat     float64
lon     float64
dtype: object
Waypoints construidos: 147. Primeros 10 (si existen):
1 {'name': 'San Juan Diego', 'type': 'Albergue', 'lat': 19.8433522, 'lon': -99.1909054}
2 {'name': 'Casa del Migrante en Tijuana', 'type': 'Albergue', 'lat': 32.5229482, 'lon': -117.0017929}
3 {'name': 'Casa del Peregrino Migrante', 'type': 'Albergue', 'lat': 20.3685143, 'lon': -99.6923611}
4 {'name': 'San Juan Diego y San Francisco de Asís', 'type': 'Albergue', 'lat': 25.8370289, 'lon': -97.4860038}
5 {'name': 'Hidalgo (Tamaulipas)', 'type': 'Frontera', 'lat': 26.0832, 'lon': -98.2773}
6 {'name': 'Templo Embajadores de Jesús', 'type': 'Albergue', 'lat': 32.5078566, 'lon': -117.0797079}
7 {'name': 'Desayunador Padre Chava', 'type': 'Comedor', 'lat': 32.5367559, 'lon': -117.032126}
8 {'name': 'Sembrando Esperanza', 'type': 'Albergue', 'lat': 22.7640846, 'lon': -102.658088}
9 {'name': 'Hotel de los Migrantes Ángeles sin Fronteras', 'type': 'Albergue', 'lat':

# **Red vial **

Punto de Usuario

In [3]:
# Punto de inicio de prueba
start = start_lat, start_lon = 19.325521, -99.167807 #cdmx

#24.497440, -107.145388 (Mazatlán)

Calcular la ONG mas cercana

In [4]:
def ong_mas_cercana(pos_actual, waypoints):
    """
    Retorna la ONG más cercana a la posición actual,
    sin importar su ubicación (norte, sur, etc.)
    y que no sea frontera.
    """
    # Filtrar solo ONGs (excluyendo fronteras)
    ongs = [
        w for w in waypoints
        if str(w['type']).strip().lower() != 'frontera'
    ]

    if not ongs:
        return None

    # Calcular la distancia geodésica de cada ONG respecto al usuario
    for o in ongs:
        o['distancia'] = geodesic(pos_actual, (o['lat'], o['lon'])).kilometers

    # Devolver la ONG más cercana
    return min(ongs, key=lambda x: x['distancia'])

siguiente = ong_mas_cercana(start, waypoints)

if siguiente:
    print(f"ONG más cercana: {siguiente['name']} ({siguiente['distancia']:.2f} km)")
else:
    print("No se encontró ninguna ONG.")


ONG más cercana: Casa Tochán (8.55 km)


Descargar la vialidad

In [5]:
# Usuario y ONG más cercana
start_point = start  # start ya es (lat, lon)
dest_point = (siguiente['lat'], siguiente['lon'])  # 'siguiente' es la ONG más cercana

# Distancia geodésica
distance_km = geodesic(start_point, dest_point).km

# Definir buffer en metros (distancia + margen extra)
buffer_m = (distance_km + 2) * 1000  # +2 km de margen extra

# Descargar grafo solo dentro del buffer desde el punto de inicio
G = ox.graph_from_point(start_point, dist=buffer_m, network_type="drive")

print("Grafo descargado con", len(G.nodes), "nodos y", len(G.edges), "aristas.")

c:\Users\samic\AppData\Local\Programs\Python\Python311\Lib\site-packages\osmnx\graph.py:191: FutureWarning: The expected order of coordinates in `bbox` will change in the v2.0.0 release to `(left, bottom, right, top)`.
  G = graph_from_bbox(


Grafo descargado con 60006 nodos y 135399 aristas.


Heuristica haversine

In [6]:
def haversine_heuristic(u, v, G):
    """
    Devuelve la distancia Haversine entre dos nodos de G.
    u, v: nodos
    G: grafo OSMnx con atributos 'y' (lat) y 'x' (lon)
    """
    lat1, lon1 = G.nodes[u]['y'], G.nodes[u]['x']
    lat2, lon2 = G.nodes[v]['y'], G.nodes[v]['x']
    R = 6371000  # Radio de la Tierra en metros
    phi1 = math.radians(lat1)
    phi2 = math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)
    a = math.sin(dphi/2)**2 + math.cos(phi1) * math.cos(phi2) * math.sin(dlambda/2)**2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1-a))
    return R * c


Encontrar nodos más cercanos al usuario y ONG

In [7]:
# Nodo más cercano al usuario
orig_node = ox.distance.nearest_nodes(G, start_lon, start_lat)  # (lon, lat)

# Nodo más cercano a la ONG
dest_node = ox.distance.nearest_nodes(G, dest_point[1], dest_point[0])


A* usando la heurística Haversine

In [8]:
route = nx.astar_path(
    G,
    orig_node,
    dest_node,
    heuristic=lambda u, v: haversine_heuristic(u, v, G),
    weight='length'
)

print("Ruta calculada con", len(route), "nodos.")


Ruta calculada con 137 nodos.


Uso de RAM

In [9]:
tracemalloc.start()  # iniciar seguimiento de memoria

route = nx.astar_path(
    G,
    orig_node,
    dest_node,
    heuristic=lambda u, v: haversine_heuristic(u, v, G),
    weight='length'
)

current, peak = tracemalloc.get_traced_memory()
print(f"Memoria usada durante el cálculo: {current / 1024:.2f} KB (actual), {peak / 1024:.2f} KB (pico)")

tracemalloc.stop()


Memoria usada durante el cálculo: 7.29 KB (actual), 1044.19 KB (pico)


consulta de las tablas con riesgos

In [10]:
# --- CONSULTA SQL ---
QUERY_FECHA = "SELECT * FROM public.fecha;"       # Ajusta el esquema si es necesario
QUERY_MUNICIPIO = "SELECT * FROM public.municipio;"

# --- CONEXIÓN Y LECTURA ---
try:
    conn = psycopg2.connect(**DB_CONFIG)
    
    # Leer tabla 'fecha'
    df_fecha = pd.read_sql(QUERY_FECHA, conn)
    print("✅ Tabla 'fecha' cargada exitosamente")
    
    # Leer tabla 'municipio'
    df_municipio = pd.read_sql(QUERY_MUNICIPIO, conn)
    print("✅ Tabla 'municipio' cargada exitosamente")
    
    conn.close()
except Exception as e:
    raise RuntimeError(f"❌ Error al conectar o leer la base de datos: {e}")

# --- Inspección rápida ---
print("Encabezados de 'fecha':", list(df_fecha.columns))
print(df_fecha.head(), "\n")

print("Encabezados de 'municipio':", list(df_municipio.columns))
print(df_municipio.head())


✅ Tabla 'fecha' cargada exitosamente
✅ Tabla 'municipio' cargada exitosamente
Encabezados de 'fecha': ['id_fecha', 'id_municipio', 'fecha', 'robos', 'secuestros', 'grado']
   id_fecha  id_municipio       fecha  robos  secuestros grado
0      5417         22001  2025-07-01     11           0  Bajo
1      2114         16033  2025-07-01      0           0  Bajo
2      6701         30126  2025-07-01      3           0  Bajo
3      2379         17009  2025-05-01     33           0  Bajo
4      3016         19040  2025-06-01      0           0  Bajo 

Encabezados de 'municipio': ['id_municipio', 'nom_municipio', 'nom_estado']
   id_municipio         nom_municipio                       nom_estado
0         30126        Paso de Ovejas  Veracruz de Ignacio de la Llave
1         20285  San Miguel Tlacamama                           Oaxaca
2         20192    San Juan Chilateca                           Oaxaca
3         24043          Tierra Nueva                  San Luis Potosí
4         15124  

C:\Users\samic\AppData\Local\Temp\ipykernel_19980\1588463305.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_fecha = pd.read_sql(QUERY_FECHA, conn)
C:\Users\samic\AppData\Local\Temp\ipykernel_19980\1588463305.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_municipio = pd.read_sql(QUERY_MUNICIPIO, conn)


Preparar los datos del último mes

In [11]:
# Convertimos fecha a datetime y filtramos último mes
df_fecha['fecha'] = pd.to_datetime(df_fecha['fecha'])
ultimo_mes = df_fecha['fecha'].max().month
ultimo_ano = df_fecha['fecha'].max().year
df_ultimo = df_fecha[(df_fecha['fecha'].dt.month == ultimo_mes) &
                     (df_fecha['fecha'].dt.year == ultimo_ano)]
df_ultimo_grado = df_ultimo[['id_municipio', 'grado']]

Buscar la siguiente ONG hacia el norte

In [ ]:
def find_sorted_ongs(start, df, min_lat=None):
    candidates = []

    for _, row in df.iterrows():
        if str(row['type']).strip().lower() != 'frontera':
            # Filtrar por latitud si se indica
            if min_lat is None or row["lat"] > min_lat:
                ong_point = (row["lat"], row["lon"])
                dist = geodesic(start, ong_point).km
                candidates.append({
                    "name": row["name"],
                    "lat": row["lat"],
                    "lon": row["lon"],
                    "type": row["type"],
                    "distancia": dist
                })

    # Ordenar por distancia
    candidates.sort(key=lambda x: x['distancia'])
    return candidates


In [ ]:
# Obtener ONGs ordenadas por distancia desde el usuario
ongs_ordenadas = find_sorted_ongs(start, df_norm)  # df_norm es tu DataFrame o ajusta a waypoints si quieres

if len(ongs_ordenadas) == 0:
    print("No hay ONGs disponibles. Dirígete a la frontera.")
elif len(ongs_ordenadas) == 1:
    ong = ongs_ordenadas[0]
    print(f"Siguiente ONG: {ong['name']} ({ong['distancia']:.1f} km) - Tipo: {ong['type']}")
else:
    # Elegir la segunda ONG más cercana
    segunda = ongs_ordenadas[1]
    print(f"Siguiente recomendación: {segunda['name']} ({segunda['distancia']:.1f} km) - Tipo: {segunda['type']}")

Siguiente recomendación: Casa de los Amigos (12.5 km) - Tipo: Albergue


**Dibujar el grafo y la ruta**

In [44]:
# --- 1️⃣ ENCONTRAR ONG MÁS CERCANA (SI NO ESTÁ DEFINIDA) ---
if 'siguiente' not in locals() and 'ong_cercana' not in locals():
    print("🔍 Buscando ONG más cercana...")
    
    def ong_mas_cercana(pos_actual, waypoints):
        """
        Retorna la ONG más cercana a la posición actual
        """
        # Filtrar solo ONGs (excluyendo fronteras)
        ongs = [
            w for w in waypoints
            if str(w.get('type', '')).strip().lower() != 'frontera'
        ]

        if not ongs:
            return None

        # Calcular la distancia geodésica de cada ONG respecto al usuario
        for o in ongs:
            o['distancia'] = geodesic(pos_actual, (o['lat'], o['lon'])).kilometers

        # Devolver la ONG más cercana
        return min(ongs, key=lambda x: x['distancia'])

    # Calcular ONG más cercana
    siguiente = ong_mas_cercana(start, waypoints)
    
    if siguiente:
        print(f"✅ ONG más cercana encontrada: {siguiente['name']} ({siguiente['distancia']:.2f} km)")
        ong_cercana = siguiente
    else:
        print("❌ No se encontró ninguna ONG cercana")
        # Crear una ONG dummy para evitar errores
        ong_cercana = {
            'name': 'ONG no disponible',
            'type': 'No disponible', 
            'lat': start[0] + 0.01,
            'lon': start[1] + 0.01,
            'distancia': 0,
            'municipio': 'Desconocido'
        }
else:
    # Si ya existe 'siguiente', úsala
    ong_cercana = siguiente if 'siguiente' in locals() else ong_cercana

# --- 2️⃣ CARGAR DATOS DE ONGs CON MUNICIPIO DESDE LA BASE DE DATOS ---

# Modificamos la consulta para obtener el nombre del municipio
QUERY_ONG = """
SELECT o.nom_ong, o.tipo, o.latitud, o.longitud, m.nom_municipio 
FROM public.ongs o
JOIN public.municipio m ON o.id_municipio = m.id_municipio;
"""

try:
    conn = psycopg2.connect(**DB_CONFIG)
    df_ongs = pd.read_sql(QUERY_ONG, conn)
    conn.close()
    print("✅ ONGs cargadas con información de municipio")
except Exception as e:
    print(f"❌ Error al cargar ONGs: {e}")
    # Si falla, cargar sin municipio
    QUERY_ONG = "SELECT nom_ong, tipo, latitud, longitud FROM public.ongs;"
    conn = psycopg2.connect(**DB_CONFIG)
    df_ongs = pd.read_sql(QUERY_ONG, conn)
    conn.close()
    df_ongs['nom_municipio'] = 'Desconocido'

# Normalizar los nombres de las columnas
df_ongs.rename(columns={
    'nom_ong': 'name',
    'tipo': 'type', 
    'latitud': 'lat',
    'longitud': 'lon',
    'nom_municipio': 'municipio'
}, inplace=True)

# Convertir a float las coordenadas
df_ongs['lat'] = pd.to_numeric(df_ongs['lat'], errors='coerce')
df_ongs['lon'] = pd.to_numeric(df_ongs['lon'], errors='coerce')

# Eliminar filas sin coordenadas
df_ongs = df_ongs.dropna(subset=['lat', 'lon'])

# Crear la lista de waypoints
waypoints = []
for _, row in df_ongs.iterrows():
    waypoints.append({
        'name': row['name'],
        'type': row['type'],
        'lat': row['lat'],
        'lon': row['lon'],
        'municipio': row['municipio']
    })

print(f"🗺️ {len(waypoints)} waypoints cargados con municipio")

# --- 3️⃣ Obtener datos de riesgo de municipios ---
# Asegurarnos de tener los datos de riesgo cargados
try:
    conn = psycopg2.connect(**DB_CONFIG)
    
    # Leer tabla 'fecha' y 'municipio'
    QUERY_FECHA = "SELECT * FROM public.fecha;"
    QUERY_MUNICIPIO = "SELECT * FROM public.municipio;"
    
    df_fecha = pd.read_sql(QUERY_FECHA, conn)
    df_municipio = pd.read_sql(QUERY_MUNICIPIO, conn)
    
    conn.close()
    
    # Procesar datos de riesgo
    df_fecha['fecha'] = pd.to_datetime(df_fecha['fecha'])
    ultimo_mes = df_fecha['fecha'].max().month
    ultimo_ano = df_fecha['fecha'].max().year
    
    df_ultimo = df_fecha[(df_fecha['fecha'].dt.month == ultimo_mes) &
                         (df_fecha['fecha'].dt.year == ultimo_ano)]
    
    # Crear diccionario de riesgo por municipio (por ID)
    riesgo_por_municipio_id = dict(zip(df_ultimo['id_municipio'], df_ultimo['grado']))
    
    # Crear diccionario de riesgo por nombre de municipio
    riesgo_por_municipio_nombre = {}
    for _, row in df_ultimo.iterrows():
        id_municipio = row['id_municipio']
        municipio_nombre = df_municipio[df_municipio['id_municipio'] == id_municipio]['nom_municipio'].iloc[0]
        riesgo_por_municipio_nombre[municipio_nombre] = row['grado']
    
    print("✅ Datos de riesgo cargados correctamente")
    
except Exception as e:
    print(f"❌ Error al cargar datos de riesgo: {e}")
    riesgo_por_municipio_nombre = {}

# --- 4️⃣ Función simplificada para determinar municipio de ruta ---
def obtener_municipio_por_proximidad(lat, lon, waypoints):
    """
    Determina municipio basado en la ONG más cercana.
    Esto es una aproximación - en producción usarías shapefiles.
    """
    min_dist = float('inf')
    municipio_cercano = 'Desconocido'
    
    for ong in waypoints:
        dist = geodesic((lat, lon), (ong['lat'], ong['lon'])).kilometers
        if dist < min_dist:
            min_dist = dist
            municipio_cercano = ong['municipio']
    
    return municipio_cercano

# --- 5️⃣ Segmentar la ruta por municipios y colores de riesgo ---
route_coords = [(G.nodes[n]['y'], G.nodes[n]['x']) for n in route]

# Crear segmentos de ruta por municipio
segmentos_ruta = []
segmento_actual = []
municipio_actual = None

print("🔍 Segmentando ruta por municipios y nivel de riesgo...")
for i, coord in enumerate(route_coords):
    lat, lon = coord
    municipio = obtener_municipio_por_proximidad(lat, lon, waypoints)
    
    # Obtener riesgo para este municipio
    riesgo = riesgo_por_municipio_nombre.get(municipio, 'Desconocido')
    
    if municipio_actual is None:
        municipio_actual = municipio
        segmento_actual.append(coord)
        riesgo_actual = riesgo
    elif municipio == municipio_actual:
        segmento_actual.append(coord)
    else:
        # Cambio de municipio - guardar segmento anterior y empezar nuevo
        if segmento_actual:
            segmentos_ruta.append({
                'coords': segmento_actual.copy(),
                'municipio': municipio_actual,
                'grado_riesgo': riesgo_actual
            })
        
        municipio_actual = municipio
        segmento_actual = [coord]
        riesgo_actual = riesgo

# Añadir el último segmento
if segmento_actual:
    segmentos_ruta.append({
        'coords': segmento_actual,
        'municipio': municipio_actual,
        'grado_riesgo': riesgo_actual
    })

print(f"📊 Ruta segmentada en {len(segmentos_ruta)} tramos por nivel de riesgo")

# --- 6️⃣ SISTEMA DE RECOMENDACIÓN DE ONGs ---
def find_sorted_ongs(start, waypoints_list, min_lat=None):
    """
    Encuentra ONGs ordenadas por distancia, excluyendo fronteras
    """
    candidates = []

    for ong in waypoints_list:
        if str(ong.get('type', '')).strip().lower() != 'frontera':
            # Filtrar por latitud si se indica
            if min_lat is None or ong["lat"] > min_lat:
                ong_point = (ong["lat"], ong["lon"])
                dist = geodesic(start, ong_point).km
                candidates.append({
                    "name": ong["name"],
                    "lat": ong["lat"],
                    "lon": ong["lon"],
                    "type": ong["type"],
                    "municipio": ong.get("municipio", "Desconocido"),
                    "distancia": dist
                })

    # Ordenar por distancia
    candidates.sort(key=lambda x: x['distancia'])
    return candidates

# Obtener ONGs ordenadas por distancia desde el usuario
print("🔍 Calculando recomendaciones de ONGs...")
ongs_ordenadas = find_sorted_ongs(start, waypoints)

# Determinar la siguiente recomendación
if len(ongs_ordenadas) == 0:
    print("❌ No hay ONGs disponibles. Dirígete a la frontera.")
    siguiente_recomendacion = None
    mensaje_recomendacion = "No hay ONGs disponibles"
elif len(ongs_ordenadas) == 1:
    siguiente_recomendacion = ongs_ordenadas[0]
    mensaje_recomendacion = f"Única ONG disponible: {siguiente_recomendacion['name']}"
    print(f"📍 Única ONG disponible: {siguiente_recomendacion['name']} ({siguiente_recomendacion['distancia']:.1f} km)")
else:
    # Elegir la segunda ONG más cercana como recomendación
    siguiente_recomendacion = ongs_ordenadas[1]
    mensaje_recomendacion = f"Recomendación: {siguiente_recomendacion['name']}"
    print(f"📍 Recomendación: {siguiente_recomendacion['name']} ({siguiente_recomendacion['distancia']:.1f} km)")

# Preparar datos para mostrar en el panel
ongs_cercanas = ongs_ordenadas[:5]  # Mostrar las 5 más cercanas

# --- 7️⃣ CREAR MAPA CON RUTA DE RIESGO, PANEL DE PESTAÑAS Y RECOMENDACIONES ---
def generar_mapa_movil_con_recomendaciones(ubicacion_usuario, ong_cercana, segmentos_ruta, waypoints, id_usuario, colores_riesgo, ongs_cercanas, siguiente_recomendacion):
    """Genera HTML optimizado para móviles con panel de pestañas incluyendo recomendaciones"""
    m = folium.Map(
        location=ubicacion_usuario,
        zoom_start=13,
        tiles="CartoDB positron",
        width='100%', 
        height='98vh'
    )
    
    # Configuración para móviles
    m.options['touchZoom'] = True
    m.options['dragging'] = True
    m.options['scrollWheelZoom'] = False
    
    # --- DIBUJAR SEGMENTOS DE RUTA CON COLORES DE RIESGO ---
    print("🎨 Dibujando ruta con colores de riesgo...")
    for i, segmento in enumerate(segmentos_ruta):
        color = colores_riesgo.get(segmento['grado_riesgo'], 'gray')
        
        folium.PolyLine(
            segmento['coords'],
            color=color,
            weight=8,
            opacity=0.9,
            tooltip=f"🏙️ {segmento['municipio']} | 🎯 Riesgo: {segmento['grado_riesgo']}"
        ).add_to(m)
        print(f"   📍 Segmento {i+1}: {segmento['municipio']} - {segmento['grado_riesgo']} ({color})")
    
    # --- MARCADOR DEL USUARIO ---
    folium.Marker(
        location=ubicacion_usuario,
        popup=folium.Popup(
            f"""
            <div style='font-size:14px; max-width:250px;'>
                <div style='background:#4A00E0; color:white; padding:8px; border-radius:5px 5px 0 0; margin:-10px -10px 10px -10px;'>
                    <b>📍 Tu Ubicación Actual</b>
                </div>
                <p><b>👤 Usuario:</b> ID {id_usuario}</p>
                <p><b>🎯 Destino:</b> {ong_cercana.get('name', 'No disponible')}</p>
            </div>
            """,
            max_width=300
        ),
        tooltip="Tu ubicación actual",
        icon=folium.Icon(color="blue", icon="user", prefix="fa")
    ).add_to(m)
    
    # --- MARCADOR DE LA ONG DESTINO CON FICHA COMPLETA ---
    if ong_cercana and ong_cercana.get('name') != 'ONG no disponible':
        municipio_ong = ong_cercana.get('municipio', 'Desconocido')
        
        # Determinar icono según tipo
        tipo_icono = {
            'Albergue': 'bed',
            'Comedor': 'utensils',
            'Frontera': 'flag',
            'default': 'home'
        }
        icono = tipo_icono.get(ong_cercana.get('type', ''), tipo_icono['default'])
        
        folium.Marker(
            location=(ong_cercana['lat'], ong_cercana['lon']),
            popup=folium.Popup(
                f"""
                <div style='font-size:14px; max-width:280px;'>
                    <div style='background:#27ae60; color:white; padding:8px; border-radius:5px 5px 0 0; margin:-10px -10px 10px -10px;'>
                        <b>🏠 ONG Destino</b>
                    </div>
                    <p><b>📌 Nombre:</b> {ong_cercana['name']}</p>
                    <p><b>🎯 Tipo:</b> {ong_cercana['type']}</p>
                    <p><b>🏙️ Municipio:</b> {municipio_ong}</p>
                    <p><b>📏 Distancia:</b> {ong_cercana['distancia']:.1f} km</p>
                    <div style='background:#f8f9fa; padding:5px; border-radius:3px; margin:5px 0;'>
                        <small>📍 {ong_cercana['lat']:.4f}, {ong_cercana['lon']:.4f}</small>
                    </div>
                </div>
                """,
                max_width=320
            ),
            tooltip=f"🎯 ONG Destino: {ong_cercana['name']}",
            icon=folium.Icon(color="green", icon=icono, prefix="fa")
        ).add_to(m)
    
    # --- MARCADOR DE LA SIGUIENTE RECOMENDACIÓN ---
    if siguiente_recomendacion:
        folium.Marker(
            location=(siguiente_recomendacion['lat'], siguiente_recomendacion['lon']),
            popup=folium.Popup(
                f"""
                <div style='font-size:14px; max-width:280px;'>
                    <div style='background:#FF9800; color:white; padding:8px; border-radius:5px 5px 0 0; margin:-10px -10px 10px -10px;'>
                        <b>⭐ Próxima Recomendación</b>
                    </div>
                    <p><b>📌 Nombre:</b> {siguiente_recomendacion['name']}</p>
                    <p><b>🎯 Tipo:</b> {siguiente_recomendacion['type']}</p>
                    <p><b>🏙️ Municipio:</b> {siguiente_recomendacion.get('municipio', 'Desconocido')}</p>
                    <p><b>📏 Distancia:</b> {siguiente_recomendacion['distancia']:.1f} km</p>
                    <div style='background:#fff3e0; padding:5px; border-radius:3px; margin:5px 0;'>
                        <small>💡 Recomendación del sistema</small>
                    </div>
                </div>
                """,
                max_width=320
            ),
            tooltip=f"⭐ Recomendación: {siguiente_recomendacion['name']}",
            icon=folium.Icon(color="orange", icon="star", prefix="fa")
        ).add_to(m)
    
    # --- OTRAS ONGs CON FICHAS INFORMATIVAS COMPLETAS ---
    ongs_marcadas = 0
    for ong in waypoints:
        if ong_cercana and ong['name'] != ong_cercana.get('name', '') and (not siguiente_recomendacion or ong['name'] != siguiente_recomendacion.get('name', '')):
            municipio_ong = ong.get('municipio', 'Desconocido')
            
            # Determinar color según tipo
            color_ong = {
                'Albergue': 'lightblue',
                'Comedor': 'orange',
                'Frontera': 'red',
                'default': 'gray'
            }
            color = color_ong.get(ong.get('type', ''), color_ong['default'])
            
            folium.CircleMarker(
                location=(ong['lat'], ong['lon']),
                radius=8,
                popup=folium.Popup(
                    f"""
                    <div style='font-size:13px; max-width:260px;'>
                        <div style='background:{color}; color:white; padding:6px; border-radius:5px 5px 0 0; margin:-10px -10px 8px -10px;'>
                            <b>🏠 Punto de Ayuda</b>
                        </div>
                        <p><b>📌 Nombre:</b> {ong['name']}</p>
                        <p><b>🎯 Tipo:</b> {ong['type']}</p>
                        <p><b>🏙️ Municipio:</b> {municipio_ong}</p>
                        <div style='background:#f8f9fa; padding:3px; border-radius:3px; margin:3px 0;'>
                            <small>📍 {ong['lat']:.4f}, {ong['lon']:.4f}</small>
                        </div>
                    </div>
                    """,
                    max_width=300
                ),
                tooltip=f"{ong['type']}: {ong['name']}",
                color=color,
                fillColor=color,
                weight=2,
                fillOpacity=0.7
            ).add_to(m)
            ongs_marcadas += 1
    
    print(f"📍 Marcadas {ongs_marcadas} ONGs adicionales con fichas informativas")
    
    # --- LEYENDA MEJORADA CON RECOMENDACIONES ---
    legend_html = '''
    <div style="
        position: fixed; 
        bottom: 20px; 
        left: 10px; 
        width: 220px; 
        height: auto;
        background-color: white; 
        border: 2px solid #4A00E0; 
        z-index: 9999; 
        font-size: 11px;
        padding: 10px;
        border-radius: 5px;
        box-shadow: 0 0 10px rgba(0,0,0,0.3);
    ">
        <h4 style="margin:0 0 8px 0; color:#4A00E0; font-size:12px;">🗺️ Leyenda del Mapa</h4>
        
        <div style="margin:5px 0;">
            <p style="margin:2px 0; font-weight:bold;">🎯 Niveles de Riesgo:</p>
            <p style="margin:2px 0;"><span style="color:red; font-weight:bold;">●</span> Alto</p>
            <p style="margin:2px 0;"><span style="color:orange; font-weight:bold;">●</span> Medio</p>
            <p style="margin:2px 0;"><span style="color:green; font-weight:bold;">●</span> Bajo</p>
            <p style="margin:2px 0;"><span style="color:gray; font-weight:bold;">●</span> Desconocido</p>
        </div>
        
        <div style="margin:5px 0;">
            <p style="margin:2px 0; font-weight:bold;">📍 Marcadores:</p>
            <p style="margin:2px 0;"><span style="color:orange;">⭐</span> Recomendación</p>
            <p style="margin:2px 0;"><span style="color:lightblue;">●</span> Albergue</p>
            <p style="margin:2px 0;"><span style="color:orange;">●</span> Comedor</p>
            <p style="margin:2px 0;"><span style="color:red;">●</span> Frontera</p>
        </div>
    </div>
    '''
    m.get_root().html.add_child(folium.Element(legend_html))
    
    # --- PANEL CON PESTAÑAS INCLUYENDO RECOMENDACIÓN ---
    destino_nombre = ong_cercana.get('name', 'No disponible') if ong_cercana else 'No disponible'
    destino_distancia = ong_cercana.get('distancia', 0) if ong_cercana else 0
    destino_tipo = ong_cercana.get('type', 'No disponible') if ong_cercana else 'No disponible'
    destino_municipio = ong_cercana.get('municipio', 'Desconocido') if ong_cercana else 'Desconocido'
    
    # Calcular estadísticas de riesgo de la ruta
    riesgo_alto = sum(1 for s in segmentos_ruta if s['grado_riesgo'] == 'Alto')
    riesgo_medio = sum(1 for s in segmentos_ruta if s['grado_riesgo'] == 'Medio')
    riesgo_bajo = sum(1 for s in segmentos_ruta if s['grado_riesgo'] == 'Bajo')
    
    # Preparar datos de recomendación
    if siguiente_recomendacion:
        rec_nombre = siguiente_recomendacion['name']
        rec_distancia = siguiente_recomendacion['distancia']
        rec_tipo = siguiente_recomendacion['type']
        rec_municipio = siguiente_recomendacion.get('municipio', 'Desconocido')
    else:
        rec_nombre = "No disponible"
        rec_distancia = 0
        rec_tipo = "No disponible"
        rec_municipio = "Desconocido"
    
    # Crear HTML para otras ONGs cercanas
    otras_ongs_html = ""
    if len(ongs_cercanas) > 2:
        for ong in ongs_cercanas[2:7]:
            otras_ongs_html += f"""
            <div style="font-size:10px; margin:4px 0; padding:5px; background:#f8f9fa; border-radius:4px; border-left: 3px solid #4A00E0;">
                <div style="font-weight:bold;">{ong['name']}</div>
                <div style="color:#666; font-size:9px;">{ong['type']} - {ong.get('municipio', 'Desconocido')} - {ong['distancia']:.1f} km</div>
            </div>
            """
    else:
        otras_ongs_html = '<div style="font-size:10px; color:#666; text-align:center;">No hay más ONGs cercanas</div>'
    
    # Crear HTML para municipios en ruta
    municipios_html = ""
    for segmento in segmentos_ruta:
        color = colores_riesgo.get(segmento['grado_riesgo'], 'gray')
        municipios_html += f'<div style="font-size:10px; margin:3px 0; padding:3px; border-left: 3px solid {color}; background: #f8f9fa;">{segmento["municipio"]} <span style="float:right; color:{color};">{segmento["grado_riesgo"]}</span></div>'
    
    info_html = f'''
<div style="
    position: fixed; 
    top: 10px; 
    right: 10px; 
    z-index: 9999; 
    font-family: Arial, sans-serif;
">
    <!-- Botón principal -->
    <div id="info-toggle" style="
        background: #4A00E0; 
        color: white; 
        padding: 8px 15px; 
        border-radius: 20px; 
        cursor: pointer; 
        font-size: 12px; 
        font-weight: bold;
        box-shadow: 0 2px 8px rgba(0,0,0,0.3);
        text-align: center;
        margin-bottom: 5px;
        display: flex;
        align-items: center;
        justify-content: center;
        gap: 5px;
    " onclick="toggleInfo()">
        <span>📋</span>
        <span>Información de Ruta</span>
        <span id="toggle-arrow">▼</span>
    </div>

    <!-- Panel principal -->
    <div id="info-panel" style="
        background: white; 
        border: 2px solid #4A00E0; 
        border-radius: 10px; 
        width: 320px; 
        max-height: 500px; 
        overflow: hidden;
        box-shadow: 0 4px 15px rgba(0,0,0,0.2);
        display: none;
    ">
        <!-- Pestañas -->
        <div style="display: flex; border-bottom: 1px solid #ddd; background: #f8f9fa;">
            <div class="tab-button active" onclick="switchTab('destino')" style="flex:1; padding:8px; text-align:center; cursor:pointer; border-bottom: 2px solid #4A00E0; font-size:11px;">🎯 Destino</div>
            <div class="tab-button" onclick="switchTab('riesgo')" style="flex:1; padding:8px; text-align:center; cursor:pointer; font-size:11px;">📊 Riesgo</div>
            <div class="tab-button" onclick="switchTab('recomendacion')" style="flex:1; padding:8px; text-align:center; cursor:pointer; font-size:11px;">⭐ Recomendación</div>
            <div class="tab-button" onclick="switchTab('ruta')" style="flex:1; padding:8px; text-align:center; cursor:pointer; font-size:11px;">🗺️ Ruta</div>
        </div>

        <!-- Contenido de pestañas -->
        <div style="padding: 12px; max-height: 400px; overflow-y: auto;">
            
            <!-- Pestaña Destino -->
            <div id="tab-destino" class="tab-content">
                <div style="margin-bottom: 15px;">
                    <h4 style="margin:0 0 8px 0; color:#4A00E0; font-size:13px;">🏠 ONG Destino Actual</h4>
                    <div style="background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; padding: 10px; border-radius: 6px;">
                        <p style="margin:0 0 5px 0; font-size:12px; font-weight:bold;">{destino_nombre}</p>
                        <p style="margin:0; font-size:10px; opacity:0.9;">{destino_tipo} - {destino_municipio}</p>
                    </div>
                </div>

                <div style="display: grid; grid-template-columns: 1fr 1fr; gap: 8px; margin-bottom: 10px;">
                    <div style="background: #e3f2fd; padding: 6px; border-radius: 4px; text-align: center;">
                        <div style="font-size:10px; color:#1976d2;">📏 Distancia</div>
                        <div style="font-size:12px; font-weight:bold; color:#1976d2;">{destino_distancia:.1f} km</div>
                    </div>
                    <div style="background: #e8f5e8; padding: 6px; border-radius: 4px; text-align: center;">
                        <div style="font-size:10px; color:#388e3c;">👤 Usuario</div>
                        <div style="font-size:12px; font-weight:bold; color:#388e3c;">ID {id_usuario}</div>
                    </div>
                </div>
            </div>

            <!-- Pestaña Riesgo -->
            <div id="tab-riesgo" class="tab-content" style="display: none;">
                <h4 style="margin:0 0 10px 0; color:#4A00E0; font-size:13px;">📊 Análisis de Riesgo</h4>
                
                <div style="margin-bottom: 15px;">
                    <div style="display: flex; justify-content: space-between; margin-bottom: 5px;">
                        <span style="font-size:11px; font-weight:bold;">Resumen de Segmentos:</span>
                        <span style="font-size:11px; font-weight:bold;">{len(segmentos_ruta)} total</span>
                    </div>
                    
                    <!-- Barra de progreso de riesgo -->
                    <div style="background: #f0f0f0; border-radius: 10px; height: 20px; margin-bottom: 10px; overflow: hidden;">
                        <div style="background: green; width: {(riesgo_bajo/len(segmentos_ruta))*100 if segmentos_ruta else 0}%; height: 100%; float: left;" title="Bajo: {riesgo_bajo}"></div>
                        <div style="background: orange; width: {(riesgo_medio/len(segmentos_ruta))*100 if segmentos_ruta else 0}%; height: 100%; float: left;" title="Medio: {riesgo_medio}"></div>
                        <div style="background: red; width: {(riesgo_alto/len(segmentos_ruta))*100 if segmentos_ruta else 0}%; height: 100%; float: left;" title="Alto: {riesgo_alto}"></div>
                    </div>
                    
                    <!-- Leyenda de colores -->
                    <div style="display: grid; grid-template-columns: 1fr 1fr 1fr; gap: 5px; text-align: center;">
                        <div>
                            <div style="color: green; font-size:12px;">● {riesgo_bajo}</div>
                            <div style="font-size:9px; color:#666;">Bajo</div>
                        </div>
                        <div>
                            <div style="color: orange; font-size:12px;">● {riesgo_medio}</div>
                            <div style="font-size:9px; color:#666;">Medio</div>
                        </div>
                        <div>
                            <div style="color: red; font-size:12px;">● {riesgo_alto}</div>
                            <div style="font-size:9px; color:#666;">Alto</div>
                        </div>
                    </div>
                </div>
            </div>

            <!-- Pestaña Recomendación -->
            <div id="tab-recomendacion" class="tab-content" style="display: none;">
                <h4 style="margin:0 0 10px 0; color:#4A00E0; font-size:13px;">⭐ Próxima Recomendación</h4>
                
                <div style="margin-bottom: 15px;">
                    <div style="background: linear-gradient(135deg, #FF9800 0%, #F57C00 100%); color: white; padding: 10px; border-radius: 6px; margin-bottom: 10px;">
                        <p style="margin:0 0 5px 0; font-size:12px; font-weight:bold;">{rec_nombre}</p>
                        <p style="margin:0; font-size:10px; opacity:0.9;">{rec_tipo} - {rec_municipio}</p>
                    </div>

                    <div style="display: grid; grid-template-columns: 1fr 1fr; gap: 8px; margin-bottom: 10px;">
                        <div style="background: #fff3e0; padding: 6px; border-radius: 4px; text-align: center;">
                            <div style="font-size:10px; color:#EF6C00;">📏 Distancia</div>
                            <div style="font-size:12px; font-weight:bold; color:#EF6C00;">{rec_distancia:.1f} km</div>
                        </div>
                        <div style="background: #e8f5e8; padding: 6px; border-radius: 4px; text-align: center;">
                            <div style="font-size:10px; color:#388e3c;">🎯 Tipo</div>
                            <div style="font-size:12px; font-weight:bold; color:#388e3c;">{rec_tipo}</div>
                        </div>
                    </div>

                    <div style="background: #e3f2fd; padding: 8px; border-radius: 5px; margin-bottom: 10px;">
                        <p style="margin:0; font-size:10px; color:#1976d2; font-weight:bold;">💡 Recomendación del Sistema</p>
                        <p style="margin:5px 0 0 0; font-size:9px; color:#1976d2;">Esta ONG ha sido seleccionada como tu próxima parada recomendada basada en proximidad y disponibilidad.</p>
                    </div>
                </div>

                <div style="border-top: 1px solid #eee; padding-top: 10px;">
                    <h5 style="margin:0 0 8px 0; color:#4A00E0; font-size:12px;">📍 Otras ONGs Cercanas</h5>
                    <div style="max-height: 150px; overflow-y: auto;">
                        {otras_ongs_html}
                    </div>
                </div>
            </div>

            <!-- Pestaña Ruta -->
            <div id="tab-ruta" class="tab-content" style="display: none;">
                <h4 style="margin:0 0 10px 0; color:#4A00E0; font-size:13px;">🗺️ Detalles de Ruta</h4>
                
                <div style="margin-bottom: 10px;">
                    <div style="font-size:11px; margin-bottom: 5px;"><b>ONGs disponibles:</b> {len(waypoints)}</div>
                    <div style="font-size:11px; margin-bottom: 5px;"><b>Segmentos calculados:</b> {len(segmentos_ruta)}</div>
                </div>

                <div style="max-height: 200px; overflow-y: auto; border: 1px solid #eee; border-radius: 5px; padding: 8px;">
                    <div style="font-size:11px; font-weight:bold; margin-bottom: 5px;">Municipios en ruta:</div>
                    {municipios_html}
                </div>
            </div>

        </div>
    </div>
</div>

<style>
.tab-button {{
    transition: all 0.3s ease;
}}
.tab-button:hover {{
    background: #e3f2fd;
}}
.tab-button.active {{
    background: #4A00E0;
    color: white;
}}
.tab-content {{
    animation: fadeIn 0.3s ease;
}}
@keyframes fadeIn {{
    from {{ opacity: 0; }}
    to {{ opacity: 1; }}
}}
</style>

<script>
function toggleInfo() {{
    var panel = document.getElementById('info-panel');
    var arrow = document.getElementById('toggle-arrow');
    if (panel.style.display === 'none' || panel.style.display === '') {{
        panel.style.display = 'block';
        arrow.innerHTML = '▲';
    }} else {{
        panel.style.display = 'none';
        arrow.innerHTML = '▼';
    }}
}}

function switchTab(tabName) {{
    // Ocultar todos los contenidos
    var contents = document.getElementsByClassName('tab-content');
    for (var i = 0; i < contents.length; i++) {{
        contents[i].style.display = 'none';
    }}
    
    // Remover clase active de todos los botones
    var buttons = document.getElementsByClassName('tab-button');
    for (var i = 0; i < buttons.length; i++) {{
        buttons[i].classList.remove('active');
    }}
    
    // Mostrar contenido seleccionado y activar botón
    document.getElementById('tab-' + tabName).style.display = 'block';
    event.target.classList.add('active');
}}

// Cerrar al hacer clic fuera
document.addEventListener('click', function(event) {{
    var panel = document.getElementById('info-panel');
    var button = document.getElementById('info-toggle');
    if (!panel.contains(event.target) && !button.contains(event.target)) {{
        panel.style.display = 'none';
        document.getElementById('toggle-arrow').innerHTML = '▼';
    }}
}});

// Inicializar con primera pestaña activa
document.addEventListener('DOMContentLoaded', function() {{
    switchTab('destino');
}});
</script>
'''
    m.get_root().html.add_child(folium.Element(info_html))
    
    # Guardar con meta tags para móvil
    archivo_html = f"ruta_movil_{id_usuario}.html"
    html_content = m.get_root().render()
    
    # Meta tags optimizados para móvil
    html_content = html_content.replace('<head>', '''
    <head>
        <meta charset="utf-8">
        <meta name="viewport" content="width=device-width, initial-scale=1.0, maximum-scale=1.0, user-scalable=no">
        <style>
            body { 
                margin: 0; 
                padding: 0; 
                font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;
            }
            #map { 
                position: absolute; 
                top: 0; 
                bottom: 0; 
                width: 100%; 
            }
            .leaflet-popup-content { 
                font-size: 14px; 
                line-height: 1.4;
            }
            .leaflet-control-zoom {
                margin-top: 180px !important;
            }
            .leaflet-popup-content-wrapper {
                border-radius: 8px;
                box-shadow: 0 3px 10px rgba(0,0,0,0.2);
            }
        </style>
    ''')
    
    with open(archivo_html, 'w', encoding='utf-8') as f:
        f.write(html_content)
    
    print(f"✅ Mapa con recomendaciones generado: {archivo_html}")
    return archivo_html

# --- 8️⃣ CONFIGURACIÓN FINAL Y EJECUCIÓN ---
def iniciar_servidor_local(puerto=8080):
    """Inicia un servidor local para servir el mapa"""
    def servir():
        handler = SimpleHTTPRequestHandler
        with HTTPServer(('0.0.0.0', puerto), handler) as httpd:
            print(f"🌐 Servidor local ejecutándose en: http://localhost:{puerto}")
            print(f"📱 Disponible en la red: http://{obtener_ip_local()}:{puerto}")
            httpd.serve_forever()
    
    thread = threading.Thread(target=servir, daemon=True)
    thread.start()
    return f"http://localhost:{puerto}/"

def obtener_ip_local():
    """Obtiene la IP local para acceso desde otros dispositivos"""
    try:
        import socket
        hostname = socket.gethostname()
        local_ip = socket.gethostbyname(hostname)
        return local_ip
    except:
        return "localhost"

# --- CONFIGURACIÓN DEL ID DE USUARIO (TEMPORAL) ---
if 'id_usuario_actual' not in locals():
    id_usuario_actual = 1
    print(f"👤 Usuario temporal asignado: ID {id_usuario_actual}")

# --- DEFINIR COLORES DE RIESGO ---
colores_riesgo = {
    'Alto': 'red',
    'Medio': 'orange', 
    'Bajo': 'green',
    'Desconocido': 'gray'
}

# --- GENERAR Y SERVIR EL MAPA MÓVIL CON SISTEMA DE RECOMENDACIONES ---
print("🚀 Generando mapa móvil con sistema de recomendaciones...")
archivo_mapa = generar_mapa_movil_con_recomendaciones(
    start, 
    ong_cercana, 
    segmentos_ruta, 
    waypoints, 
    id_usuario_actual,
    colores_riesgo,
    ongs_cercanas,
    siguiente_recomendacion
)

# Iniciar servidor
url_base = iniciar_servidor_local(8080)

print(f"\n{'='*60}")
print("📱 **MAPA CON SISTEMA DE RECOMENDACIONES**")
print(f"{'='*60}")
print(f"📍 URL local: {url_base}{archivo_mapa}")
print(f"🌐 URL red local: http://{obtener_ip_local()}:8080/{archivo_mapa}")
print(f"📁 Archivo: {archivo_mapa}")
print(f"🎯 ONG destino actual: {ong_cercana.get('name', 'No disponible')}")
print(f"⭐ ONG recomendada: {siguiente_recomendacion.get('name', 'No disponible') if siguiente_recomendacion else 'No disponible'}")
print(f"📏 Distancia recomendada: {siguiente_recomendacion.get('distancia', 0):.1f} km" if siguiente_recomendacion else "📏 No disponible")
print(f"📊 Segmentos de riesgo: {len(segmentos_ruta)}")
print(f"📍 ONGs en mapa: {len(waypoints)}")
print(f"{'='*60}")
print("💡 **Nuevas características:**")
print("• ⭐ Nueva pestaña de Recomendaciones")
print("• 🎯 Marcador naranja para la ONG recomendada")
print("• 📋 Lista de ONGs cercanas adicionales")
print("• 🗺️ Panel mejorado con 4 pestañas organizadas")
print("• 📱 Interfaz optimizada para móviles")
print(f"{'='*60}")

# Opcional: Abrir automáticamente en el navegador
try:
    webbrowser.open(f"{url_base}{archivo_mapa}")
    print("🌐 Abriendo mapa en navegador...")
except:
    pass

C:\Users\samic\AppData\Local\Temp\ipykernel_19980\1756848501.py:57: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_ongs = pd.read_sql(QUERY_ONG, conn)


✅ ONGs cargadas con información de municipio
🗺️ 120 waypoints cargados con municipio


C:\Users\samic\AppData\Local\Temp\ipykernel_19980\1756848501.py:107: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_fecha = pd.read_sql(QUERY_FECHA, conn)
C:\Users\samic\AppData\Local\Temp\ipykernel_19980\1756848501.py:108: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_municipio = pd.read_sql(QUERY_MUNICIPIO, conn)


✅ Datos de riesgo cargados correctamente
🔍 Segmentando ruta por municipios y nivel de riesgo...
📊 Ruta segmentada en 1 tramos por nivel de riesgo
🔍 Calculando recomendaciones de ONGs...
📍 Recomendación: Casa de los Amigos (12.5 km)
🚀 Generando mapa móvil con sistema de recomendaciones...
🎨 Dibujando ruta con colores de riesgo...
   📍 Segmento 1: Álvaro Obregón - Medio (orange)
📍 Marcadas 118 ONGs adicionales con fichas informativas
✅ Mapa con recomendaciones generado: ruta_movil_1.html

📱 **MAPA CON SISTEMA DE RECOMENDACIONES**
📍 URL local: http://localhost:8080/ruta_movil_1.html
🌐 URL red local: http://192.168.1.156:8080/ruta_movil_1.html
📁 Archivo: ruta_movil_1.html
🎯 ONG destino actual: Casa Tochán
⭐ ONG recomendada: Casa de los Amigos
📏 Distancia recomendada: 12.5 km
📊 Segmentos de riesgo: 1
📍 ONGs en mapa: 120
💡 **Nuevas características:**
• ⭐ Nueva pestaña de Recomendaciones
• 🎯 Marcador naranja para la ONG recomendada
• 📋 Lista de ONGs cercanas adicionales
• 🗺️ Panel mejorado co